# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
# Imports
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from scraper import fetch_website_contents

In [ ]:
# Initialize: load env, create client, model options (OpenAI only - switch for speed vs quality)
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

openai = OpenAI()
MODELS = {"gpt-4.1-mini": "gpt-4.1-mini", "gpt-5-nano (faster/cheaper)": "gpt-5-nano"}
current_model = ["gpt-4.1-mini"]  # default (dropdown label); updated by dropdown, mapped to id in chat()

In [ ]:
# System prompt: technical tutor expertise (Python, software eng, data science, LLMs)
system_message = """You are a helpful technical tutor. You answer questions about Python, software engineering, data science, and LLMs in a clear, concise way.
Give accurate explanations. If you don't know something, say so. Use markdown for structure (headers, lists, code snippets) when helpful.
When the user shares a URL, you may use the fetch_url_content tool to get the page content and then explain or summarize it. Use the tool when it would help answer their question."""

In [ ]:
# Tool: fetch URL content (so the assistant can read a page and explain/summarize it)
def fetch_url_content(url: str) -> str:
    """Fetch and return the text content of a webpage. Use when the user asks about a URL."""
    try:
        return fetch_website_contents(url)
    except Exception as e:
        return f"Error fetching URL: {e}"

fetch_url_function = {
    "name": "fetch_url_content",
    "description": "Fetch the text content of a webpage at the given URL. Use when the user asks to explain, summarize, or read a web page.",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {"type": "string", "description": "Full URL of the page (e.g. https://example.com)"},
        },
        "required": ["url"],
        "additionalProperties": False,
    },
}
tools = [{"type": "function", "function": fetch_url_function}]

In [ ]:
# Handle tool calls from the model (run our function and return tool results)
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "fetch_url_content":
            args = json.loads(tool_call.function.arguments)
            url = args.get("url", "")
            content = fetch_url_content(url)
            responses.append({
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call.id,
            })
    return responses

In [ ]:
# Streaming chat: conversation history + tool support (loop until model stops asking for tools)
def chat(message, history):
    model = MODELS.get(current_model[0], current_model[0])  # map dropdown label to API model id
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=model, messages=messages, tools=tools)

    # Handle tool calls in a loop (model might call tool, we send result back, it may call again)
    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        tool_responses = handle_tool_calls(msg)
        messages.append(msg)
        messages.extend(tool_responses)
        response = openai.chat.completions.create(model=model, messages=messages, tools=tools)

    # Stream the final text reply (typewriter effect): yield word by word
    final_content = response.choices[0].message.content or ""
    if not final_content:
        yield "(No text reply)"
        return
    result = ""
    for word in final_content.split():
        result += word + " "
        yield result

In [ ]:
# Gradio UI: model dropdown + chat interface (streaming, markdown, examples)
def set_model(m):
    current_model[0] = m
    return m

with gr.Blocks(title="Technical Q&A Tutor", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## Week 2 – Technical Q&A Tutor\nAsk anything about Python, software engineering, data science, or LLMs. Paste a URL and ask to explain or summarize it—the assistant can fetch the page.")
    model_selector = gr.Dropdown(
        choices=list(MODELS.keys()),
        value="gpt-4.1-mini",
        label="Model",
        info="Switch between faster/cheaper (gpt-5-nano) and stronger (gpt-4.1-mini)",
    )
    model_selector.change(fn=set_model, inputs=model_selector)
    gr.ChatInterface(
        fn=chat,
        type="messages",
        examples=[
            "Explain what a Python generator is and when to use it.",
            "What does this code do? x = [n**2 for n in range(10)]",
            "Summarize this page: https://www.python.org/about/",
        ],
    )

demo.launch()